In [39]:
import os
import albumentations as A
import cv2 as cv
import matplotlib.pyplot as plt
import pandas as pd
from uuid import uuid4 as uuid

In [ ]:
train_transforms = A.Compose([
    # --- تغییرات هندسی با پس‌زمینه سفید کاغذ ---
    A.Affine(
        scale={"x": (0.9, 1.1), "y": (0.95, 1.05)},
        translate_percent={"x": (-0.03, 0.03), "y": (-0.03, 0.03)},
        rotate=(-1, 1),
        shear={"x": (-7, 7), "y": (-2, 2)},
        cval=255,  # پر کردن لبه‌های خالی با رنگ سفید
        p=0.7,
    ),
    A.ElasticTransform(
        alpha=15,          # عدد ۱۵ بسیار متعادل است و نقطه‌های حروف را خراب نمی‌کند
        sigma=4, 
        border_mode=cv.BORDER_CONSTANT, 
        value=255,         # لبه‌ها سفید
        p=0.35
    ),
    A.Perspective(
        scale=(0.02, 0.04), 
        pad_mode=cv.BORDER_CONSTANT, 
        pad_val=255,       # پد سفید
        p=0.25
    ),

    # --- نوری و کیفیت تصویر ---
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(10.0, 30.0), p=0.3),
    A.MotionBlur(blur_limit=(3, 5), p=0.15),
    A.ImageCompression(quality_lower=70, quality_upper=100, p=0.2),
])


In [41]:
train_data_path = os.path.join("dataset/Farsi/train/")
test_data_path = os.path.join("dataset/Farsi/test/")

train_labels = pd.read_csv(train_data_path+"labels.csv", header=None)
test_labels = pd.read_csv(test_data_path+"labels.csv", header=None)

In [43]:
def anotator(labels, data_path):
    for i in range(len(labels)):
        file_name = labels[0][i]
        file_label = labels[1][i]
        # Read an image with OpenCV and convert it to the RGB colorspace
        image = cv.imread(f"{data_path}images/{file_name}.png")
        image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
        new_image = transform(image=image)["image"]
        new_name = file_name+str(uuid()).split("-")[0]
        save_path = data_path+"images/"+new_name+".png"
        plt.imsave(save_path, new_image)
        data = {0: [new_name], 1: [file_label]}
        df = pd.DataFrame(data)
        df.to_csv(data_path+"labels.csv", mode='a', index=False, header=False)

In [44]:
anotator(train_labels, train_data_path)
anotator(test_labels, test_data_path)